# 集群上svaba使用流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境及编译

### 1、创建环境

In [ ]:
conda create -n svaba_env -y -c conda-forge -c bioconda \
    cmake>=3.14 \
    gcc cxx-compiler \
    htslib zlib bzip2 xz \
    jemalloc sqlite

conda activate svaba_env

conda install samtools -c bioconda -y

### 2、下载svaba源码并编译 

In [ ]:
cd /mnt/home/ygjx/chenkejin/manta/

git clone --recursive https://github.com/walaj/svaba
cd svaba

sed -i '/#include <memory>/a #include <cstdint>' /mnt/home/ygjx/chenkejin/svaba/src/svaba/LearnBamParams.h

mkdir -p build && cd build
cmake .. -DUSE_JEMALLOC=ON

make -j 8

### 3、svaba要求目录下有一整套专属的 BWA 索引文件（通常包含 .bwt, .pac, .ann, .amb, .sa 这 5 个隐藏文件），可用下面的脚本构建

In [ ]:
#!/bin/bash
#SBATCH --job-name=bwa_index
#SBATCH --nodes=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --output=bwa_index.log

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba_env

echo "开始构建 BWA 索引 (预计需要 1-2 小时)..."
bwa index /mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta
echo "构建完成！"

## 三、运行

In [ ]:
mkdir -p /mnt/home/ygjx/chenkejin/SvABA
cd /mnt/home/ygjx/chenkejin/SvABA

### 1、单样本运行

### 脚本路径：/mnt/home/ygjx/chenkejin/SvABA/test_single_svaba.sh

### sbatch test_single_svaba.sh 运行

In [ ]:
#!/bin/bash
#SBATCH --job-name=svaba_1866277           # 作业名称
#SBATCH --nodes=1                          # 申请 1 个节点
#SBATCH --cpus-per-task=24                 # 单样本测试拉满 24 个 CPU
#SBATCH --mem=48G                          # 内存给足 48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/SvABA/logs/svaba_1866277_%j.log

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba_env

# --- 1. 全局变量与路径 (已完美适配你的实际绝对路径) ---
# 精准指向你的 SvABA 源码根目录
SVABA_DIR="/mnt/home/ygjx/chenkejin/svaba" 

# 参考基因组与官方自带黑名单
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
BLACKLIST="${SVABA_DIR}/tracks/hg38.combined_blacklist.bed"

# 数据源与工作空间
SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/SvABA"
PREFIX="1866277"
NORMAL_ID="${PREFIX}N"
TUMOR_ID="${PREFIX}T"
THREADS=24

# 精准锁定 BAM 文件
NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 开始处理 SvABA 测试样本: ${PREFIX}"
echo "=========================================================="

# --- 2. 建立专属沙盒目录 ---
mkdir -p "${WORK_DIR}/logs"
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_svaba_run"
rm -rf "$RUN_DIR" && mkdir -p "$RUN_DIR"
cd "$RUN_DIR"

# 注入 Conda 环境中的 jemalloc 动态库路径，极大提升多线程组装速度
export JEMALLOC_LIB="${CONDA_PREFIX}/lib/libjemalloc.so.2"
# 精准定位你刚才编译出来的绿色可执行文件
export SVABA="${SVABA_DIR}/build/svaba"

# --- 3. Step 1: 运行 svaba run (底层调用 jemalloc 加速) ---
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 运行 svaba run..."
${SVABA_DIR}/svaba_jemalloc run \
    -t "$TUMOR_BAM" \
    -n "$NORMAL_BAM" \
    -G "$REF_FA" \
    -a "$PREFIX" \
    -p "$THREADS" \
    --blacklist "$BLACKLIST"

# --- 4. Step 2: 运行 svaba postprocess (合并与去重) ---
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 svaba postprocess..."
${SVABA} postprocess \
    -i "$PREFIX" \
    -t "$THREADS" \
    -m 4G

# --- 5. Step 3: 运行 svaba tovcf (标准格式转化) ---
echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 运行 svaba tovcf..."
${SVABA} tovcf \
    -i "${PREFIX}.bps.sorted.dedup.txt.gz" \
    -b "$TUMOR_BAM" \
    -a "$PREFIX"

# --- 6. 提取终极产物与沙盒清理 ---
if [ -f "${PREFIX}.sv.vcf.gz" ]; then
    cp "${PREFIX}.sv.vcf.gz" "${FINAL_VCF_DIR}/"
    cp "${PREFIX}.indel.vcf.gz" "${FINAL_VCF_DIR}/"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 任务成功！结果已提取至: ${FINAL_VCF_DIR}"
    
    # 清理巨大的中间组装文件
    cd "${WORK_DIR}"
    rm -rf "$RUN_DIR"
else
    echo "⚠️ 错误：未找到输出的 VCF 文件，SvABA 运行可能失败。"
    exit 1
fi

### 2、批量处理

### 脚本路径：/mnt/home/ygjx/chenkejin/SvABA/batch_svaba.sh

### sbatch batch_svaba.sh 运行

In [ ]:
#!/bin/bash
#SBATCH --job-name=SvABA_Batch             # 作业名称
#SBATCH --nodes=1                          # 每个子任务申请 1 个节点
#SBATCH --cpus-per-task=16                 # 【精准分配】每个子任务分配 16 个 CPU
#SBATCH --mem=48G                          # 【内存保险】16核并发较高，内存提至 48G 防止 OOM
#SBATCH --array=1-80%20                    # 【并发拉满】共 80 对样本，每次最多同时跑 20 个 (最高占用 320 核)
#SBATCH --output=/mnt/home/ygjx/chenkejin/SvABA/logs/slurm_array_%A_%a.out 

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba_env

# --- 1. 全局变量与路径 ---
SVABA_DIR="/mnt/home/ygjx/chenkejin/svaba" 
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
BLACKLIST="${SVABA_DIR}/tracks/hg38.combined_blacklist.bed"
THREADS=16

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/SvABA"
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"

# 总体进度汇报日志
MASTER_LOG="${WORK_DIR}/master_progress_svaba.log"

# --- 2. 任务解析 (根据阵列 ID 获取当前需要处理的行) ---
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST")
NORMAL_ID=$(echo "$LINE" | awk '{print $1}')
TUMOR_ID=$(echo "$LINE" | awk '{print $2}')

# 提取公共前缀
PREFIX=$(echo "$NORMAL_ID" | sed 's/N//')

# 【防崩溃保险】读到空行直接退出
if [ -z "$PREFIX" ]; then
    exit 0
fi

# --- 3. 专属日志动态重定向 ---
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.log"
# 从此行起，该子任务的所有标准输出/错误将定向至专属 log 文件
exec > >(tee -i "$SAMPLE_LOG") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] SvABA 样本 ${PREFIX} 开始处理"
echo "任务阵列 ID: ${SLURM_ARRAY_TASK_ID}/80  执行节点: $(hostname)"
echo "分配资源: 16 CPUs, 48G Mem"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} on $(hostname)" >> "$MASTER_LOG"

# --- 4. 建立绝对隔离沙盒与结果汇总目录 ---
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

# 定义最终输出文件路径 (在汇总文件夹内)
final_sv_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.sv.vcf"
final_indel_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.indel.vcf"

# 【断点续传】如果解压后的 VCF 已在汇总文件夹中，直接跳过，绝不浪费算力
if [ -f "$final_sv_vcf" ] && [ -f "$final_indel_vcf" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] 样本 ${PREFIX} 结果已存在，跳过运行。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Sample ${PREFIX} skipped" >> "$MASTER_LOG"
    exit 0
fi

# 【核心防线】创建属于该样本的绝对独立沙盒，并在起跑前强制清空可能存在的残骸
RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_svaba_run"
rm -rf "$RUN_DIR" && mkdir -p "$RUN_DIR"
cd "$RUN_DIR"

# 注入 jemalloc 动态库 (突破多线程锁争用瓶颈)
export JEMALLOC_LIB="${CONDA_PREFIX}/lib/libjemalloc.so.2"
export SVABA="${SVABA_DIR}/build/svaba"

# --- 5. 精准锁定源数据 ---
NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# --- 6. 核心流程执行 (附带严格错误拦截机制) ---
{
    # Step 1: 组装与突变寻找
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 运行 svaba run (16核拉满)..."
    ${SVABA_DIR}/svaba_jemalloc run \
        -t "$TUMOR_BAM" \
        -n "$NORMAL_BAM" \
        -G "$REF_FA" \
        -a "$PREFIX" \
        -p "$THREADS" \
        --blacklist "$BLACKLIST"

    # Step 2: 坐标排序与去重
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 svaba postprocess..."
    ${SVABA} postprocess \
        -i "$PREFIX" \
        -t "$THREADS" \
        -m 4G

    # Step 3: 格式转化
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 运行 svaba tovcf..."
    ${SVABA} tovcf \
        -i "${PREFIX}.bps.sorted.dedup.txt.gz" \
        -b "$TUMOR_BAM" \
        -a "$PREFIX"

    # Step 4: 结果解压与汇聚
    if [ -f "${PREFIX}.sv.vcf.gz" ] && [ -f "${PREFIX}.indel.vcf.gz" ]; then
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] 正在执行解压操作并汇总至 Final_Results..."
        
        # 使用 gunzip -c 直接将沙盒内的压缩包解压为纯文本，并写出到外部的汇总文件夹
        gunzip -c "${PREFIX}.sv.vcf.gz" > "$final_sv_vcf"
        gunzip -c "${PREFIX}.indel.vcf.gz" > "$final_indel_vcf"
        
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 处理完成！"
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} completed" >> "$MASTER_LOG"
        
        # 【阅后即焚】安全离开沙盒并将其彻底摧毁，释放磁盘空间
        cd "${WORK_DIR}"
        rm -rf "$RUN_DIR"
    else
        echo "⚠️ 错误：在沙盒中未找到生成的 vcf.gz 压缩包！"
        false # 故意触发 || 后面的错误拦截逻辑
    fi

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 样本 ${PREFIX} 运行失败！"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} failed!" >> "$MASTER_LOG"
    exit 1
}

### tail -f /mnt/home/ygjx/chenkejin/SvABA/master_progress_svaba.log  可实时查看进度

### 结果路径：/mnt/home/ygjx/chenkejin/SvABA/Final_Results/